# Model: patronus-studio/wolf-defender-prompt-injection

**Our final submission.** One approved backbone, `wolf-defender-prompt-injection` (a safety-tuned ModernBERT), fine-tuned with 5-fold cross-validation, then combined by majority vote with our two earlier runs of the same model. **Leaderboard:** main-v2 alone scored 0.97069.

**How to read this notebook**

| Step | What it does | Where to look in the output |
|---|---|---|
| 1. Training script | Settings and the per-fold training loop | The settings block at the top |
| 2. Train 5 folds | Two folds at a time, one per T4 | Each fold's `done: best epoch …, val macro-F1 …` line |
| 3. Out-of-fold score | A held-out prediction for every train row | `OOF macro-F1`: our estimate of test performance |
| 4. Class offsets | Writes `submission.csv` (this run alone) | `offsets … (held-out gain …)` |
| 5. Final vote | Writes `submission_vote.csv`, our submitted file | How many rows the vote changed |

**Method in brief:** up to 384 tokens per prompt (long prompts keep their first 2/3 and last 1/3, since attacks often sit at the end); lr 3e-5, effective batch 64, 4 epochs, keeping each fold's best epoch; class weights ∝ 1/√frequency for the 62/31/7 class imbalance; the test prediction is the mean of the 5 fold models.

**Rules:** one approved model throughout; trained on `train.csv` only, with `test.csv` used only for prediction; runs offline.

**To reproduce:** GPU T4×2, Internet off. Attach the competition data, the `wolf-defender-prompt-injection` model and our dataset with the main-v2 and main-v3 submission files (their names are set in step 5), then Run All (about 35–40 min). To rerun only steps 3–5 without a GPU, attach this notebook's saved output instead.

**Step 1: training script.** Written to `train.py` so each GPU can run it as its own process. The settings block is the only part that differs between our runs (main-v2 used seed 42).

In [ ]:
%%writefile train.py
# Trains the given folds and writes their predictions; step 2 runs it once per GPU, e.g. `python train.py 0 2 4`.
import gc, glob, json, math, os, sys, time
import numpy as np, pandas as pd, torch, transformers
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

# ---- settings: identical to main-v2 except the seed ----------------------------------------------
MAX_LEN, HEAD_FRAC = 384, 2 / 3                 # long prompts keep their first 2/3 and last 1/3 of tokens
LR, EPOCHS, SEED = 3e-5, 4, 43
BS, ACCUM, WARMUP, WD = 16, 4, 0.1, 0.01        # effective batch = BS * ACCUM = 64
VARIANT = f"wolf384_s{SEED}"                    # prefix of this run's output files
MODEL_NAME, LABELS, OUT = "wolf-defender", ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"], "/kaggle/working"
CONFIG = {k: v for k, v in dict(globals()).items() if k.isupper()}

transformers.logging.set_verbosity_error()  # hides the harmless "LOAD REPORT" and long-prompt warnings
dev = torch.device("cuda")
DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
models = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if MODEL_NAME in p.lower()]
assert models, f"attach the {MODEL_NAME} model as an input"

train, test = pd.read_csv(f"{DATA_DIR}/train.csv"), pd.read_csv(f"{DATA_DIR}/test.csv")
y = train.label.map({l: i for i, l in enumerate(LABELS)}).values
fold = np.zeros(len(train), dtype=int)
for k, (_, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(train, train.label + "_" + train.language)):
    fold[va] = k
tok = AutoTokenizer.from_pretrained(models[0], local_files_only=True)
core, full = tok("hello world", add_special_tokens=False)["input_ids"], tok("hello world")["input_ids"]
cut = next(i for i in range(len(full)) if full[i:i + len(core)] == core)
PRE, POST = full[:cut], full[cut + len(core):]  # this tokenizer's own special tokens, e.g. [CLS] ... [SEP]

def encode(df):
    budget = MAX_LEN - len(PRE) - len(POST)
    head = int(budget * HEAD_FRAC)
    # long prompts keep their first `head` and last `budget - head` tokens
    return [PRE + (x if len(x) <= budget else x[:head] + x[len(x) - budget + head:]) + POST
            for x in tok(list(df.text), add_special_tokens=False)["input_ids"]]

X, X_te = encode(train), encode(test)
assert max(map(len, X + X_te)) <= MAX_LEN

def pad(seqs):
    ids = torch.full((len(seqs), max(map(len, seqs))), tok.pad_token_id)
    mask = torch.zeros_like(ids)
    for r, s in enumerate(seqs):
        ids[r, :len(s)], mask[r, :len(s)] = torch.tensor(s), 1
    return {"input_ids": ids.to(dev), "attention_mask": mask.to(dev)}

def train_batches(idx, rng):
    # sort by length inside chunks of 50 batches: little padding, order still random
    perm, out = rng.permutation(idx), []
    for i in range(0, len(perm), BS * 50):
        c = perm[i:i + BS * 50]
        c = c[np.argsort([len(X[j]) for j in c])]
        out += [c[k:k + BS] for k in range(0, len(c), BS)]
    return [out[i] for i in rng.permutation(len(out))]

@torch.no_grad()
def predict(model, seqs, bs=32):
    model.eval()
    order, out = np.argsort([len(s) for s in seqs]), np.zeros((len(seqs), 3), dtype=np.float32)
    for i in range(0, len(seqs), bs):
        b = order[i:i + bs]
        with torch.autocast("cuda", dtype=torch.float16):
            out[b] = model(**pad([seqs[j] for j in b])).logits.float().softmax(-1).cpu().numpy()
    return out

def optimizer(model):
    no_decay = lambda n: "bias" in n or "norm" in n.lower()
    return torch.optim.AdamW([
        {"params": [p for n, p in model.named_parameters() if not no_decay(n)], "lr": LR, "weight_decay": WD},
        {"params": [p for n, p in model.named_parameters() if no_decay(n)], "lr": LR, "weight_decay": 0.0}])

def run_fold(k, seed):
    gc.collect(); torch.cuda.empty_cache()
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    tr_idx, va_idx = np.where(fold != k)[0], np.where(fold == k)[0]
    model = AutoModelForSequenceClassification.from_pretrained(  # its 2-class head is replaced by a new 3-class one
        models[0], num_labels=3, ignore_mismatched_sizes=True, local_files_only=True).to(dev)
    w = 1 / np.sqrt(np.bincount(y[tr_idx], minlength=3) / len(tr_idx))
    loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(w / w.mean(), dtype=torch.float32, device=dev))
    opt = optimizer(model)
    steps = EPOCHS * math.ceil(math.ceil(len(tr_idx) / BS) / ACCUM)  # optimizer steps
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    scaler = torch.amp.GradScaler("cuda")
    langs, log, best_f1, t_start = train.language.values[va_idx], [], -1, time.time()
    print(f"[fold {k}] seed {seed}: {len(tr_idx)} train / {len(va_idx)} val rows", flush=True)
    for ep in range(1, EPOCHS + 1):
        model.train()
        t0, batches = time.time(), train_batches(tr_idx, rng)
        for i, b in enumerate(batches, 1):
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**pad([X[j] for j in b])).logits
            loss = loss_fn(logits.float(), torch.tensor(y[b], device=dev))
            scaler.scale(loss / ACCUM).backward()
            if i % ACCUM == 0 or i == len(batches):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            if i == 50 or i % 400 == 0:
                print(f"[fold {k}] ep{ep} step {i}/{len(batches)} loss {loss.item():.4f} | "
                      f"{(time.time() - t0) / i * (len(batches) - i) / 60:.1f} min left in epoch", flush=True)
        p = predict(model, [X[j] for j in va_idx])
        pred, yv = p.argmax(1), y[va_idx]
        f1 = f1_score(yv, pred, average="macro")
        log.append({"epoch": ep, "macro_f1": f1, "minutes": (time.time() - t_start) / 60,
                    "per_class_f1": dict(zip(LABELS, f1_score(yv, pred, average=None, labels=[0, 1, 2]).tolist())),
                    "per_lang_f1": {l: f1_score(yv[langs == l], pred[langs == l], average="macro") for l in np.unique(langs)}})
        print(f"[fold {k}] == epoch {ep}: val macro-F1 {f1:.4f} ({(time.time() - t0) / 60:.1f} min)", flush=True)
        if ep == 1 and f1 < 0.40:
            return None  # collapsed to (nearly) one class
        if f1 > best_f1:  # keep the best epoch's weights in CPU RAM, never on disk
            best_f1, best_ep, best_p = f1, ep, p
            best_state = {n: t.to("cpu", copy=True) for n, t in model.state_dict().items()}
    model.load_state_dict(best_state)
    np.savez(f"{OUT}/oof_{VARIANT}_f{k}.npz", ids=np.asarray(train.id, dtype=str)[va_idx], probs=best_p)
    np.savez(f"{OUT}/test_{VARIANT}_f{k}.npz", ids=np.asarray(test.id, dtype=str), probs=predict(model, X_te))
    with open(f"{OUT}/log_{VARIANT}_f{k}.json", "w") as f:
        json.dump({"variant": VARIANT, "fold": k, "seed": seed, "config": CONFIG, "epochs": log, "best_epoch": best_ep}, f, indent=1)
    print(f"[fold {k}] done: best epoch {best_ep}/{EPOCHS}, val macro-F1 {best_f1:.4f}", flush=True)
    return best_f1

folds_to_run = [int(a) for a in sys.argv[1:]]
if 0 in folds_to_run:  # only one process writes the shared fold file
    pd.DataFrame({"id": train.id, "fold": fold}).to_csv(f"{OUT}/folds.csv", index=False)
for k in folds_to_run:
    if run_fold(k, SEED) is None:
        print(f"[fold {k}] collapsed (epoch-1 macro-F1 < 0.40): retrying with seed {SEED + 1000}", flush=True)
        if run_fold(k, SEED + 1000) is None:
            sys.exit(f"[fold {k}] collapsed twice: stop and check the setup")

**Step 2: train the 5 folds** (about 30–35 min). Folds 0, 2, 4 run on one T4 and folds 1, 3 on the other. Each fold prints its validation macro-F1 after every epoch and keeps its best epoch.

In [ ]:
import os, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation with mixed batch lengths
if torch.cuda.device_count() >= 2:  # one process per T4: folds 0, 2, 4 and 1, 3
    !CUDA_VISIBLE_DEVICES=0 python -u train.py 0 2 4 & CUDA_VISIBLE_DEVICES=1 python -u train.py 1 3 & wait
else:
    !python -u train.py 0 1 2 3 4

**Step 3: out-of-fold score.** Joins the 5 folds' validation predictions, so every train row gets a prediction from a model that never saw it. This macro-F1 is our estimate of test performance.

In [ ]:
import glob, os
import numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, f1_score

LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
train, test = pd.read_csv(f"{DATA_DIR}/train.csv"), pd.read_csv(f"{DATA_DIR}/test.csv")
y = train.label.map({l: i for i, l in enumerate(LABELS)}).values

# this session's fold files, or this notebook's saved output attached as an input
find = lambda kind: sorted(glob.glob(f"/kaggle/working/{kind}_*_f[0-9].npz") or glob.glob(f"/kaggle/input/**/{kind}_*_f[0-9].npz", recursive=True))
oof, tst = [np.load(p) for p in find("oof")], [np.load(p) for p in find("test")]
assert len(oof) == len(tst) == 5, f"found {len(oof)} OOF and {len(tst)} test fold files, need 5 of each: run step 2, or attach this notebook's saved output"
ids = np.concatenate([d["ids"] for d in oof])
assert len(ids) == len(set(ids)) == len(train) and set(ids) == set(train.id), "OOF ids missing or duplicated"
assert all((d["ids"] == np.asarray(test.id, dtype=str)).all() for d in tst), "test rows out of order"
P = pd.DataFrame(np.concatenate([d["probs"] for d in oof]), index=ids).loc[train.id].values  # one OOF prediction per train row
T = np.mean([d["probs"] for d in tst], axis=0)  # test prediction = mean of the 5 fold models
pred = P.argmax(1)
print(f"OOF macro-F1 {f1_score(y, pred, average='macro'):.4f} | per class",
      dict(zip(LABELS, f1_score(y, pred, average=None).round(4).tolist())))
print("per language:", {l: round(f1_score(y[m], pred[m], average="macro"), 4) for l, m in train.groupby("language").indices.items()})
print(confusion_matrix(y, pred))

**Step 4: class offsets → `submission.csv`.** Tries small shifts to the JAILBREAK and PROMPT_INJECTION scores on the out-of-fold predictions, and keeps them only if they also improve held-out rows; otherwise both stay 0.

In [ ]:
import itertools
from sklearn.model_selection import StratifiedKFold

macro = lambda yy, scores: f1_score(yy, scores.argmax(1), average="macro")
logp, logt = np.log(P + 1e-9), np.log(T + 1e-9)

def tune(lp, yy, step):
    # class offsets on log-probs; checked from (0, 0) outwards, so a tie keeps the smallest offsets
    g = np.round(np.arange(-1.5, 1.5001, step), 2)
    return max(sorted(itertools.product(g, g), key=lambda b: abs(b[0]) + abs(b[1])), key=lambda b: macro(yy, lp + [0, *b]))

# keep offsets only if they also help on held-out rows: tune on 4/5 of the OOF rows, score on the other 1/5
gains = []
for a, b in StratifiedKFold(5, shuffle=True, random_state=0).split(logp, y):
    gains.append(macro(y[b], logp[b] + [0, *tune(logp[a], y[a], 0.1)]) - macro(y[b], logp[b]))
OFFSET = np.array([0, *tune(logp, y, 0.05)]) if np.mean(gains) > 0 else np.zeros(3)

pred = (logt + OFFSET).argmax(1)
sub = pd.DataFrame({"id": test.id, "label": np.array(LABELS)[pred]})
assert len(sub) == 6000 and sub.id.is_unique and set(sub.id) == set(test.id) and sub.label.isin(LABELS).all()
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"OOF macro-F1 raw {macro(y, logp):.4f} -> with offsets {macro(y, logp + OFFSET):.4f} | "
      f"offsets JB {OFFSET[1]:+.2f} PI {OFFSET[2]:+.2f} (held-out gain {np.mean(gains):+.4f}) | "
      f"test share {(np.bincount(pred, minlength=3) / len(pred)).round(3).tolist()}")

**Step 5: final vote → `submission_vote.csv` (our submitted file).** A 2-of-3 majority vote between this run and our earlier main-v2 and main-v3 runs of the same model; where all three disagree, this run decides. A guard stops the vote if the attached files don't look like those two runs.

In [ ]:
EARLIER = {"main-v2": "submission-1.csv", "main-v3": "submission-3.csv"}  # file names in our attached dataset
paths = {run: glob.glob(f"/kaggle/input/**/{f}", recursive=True) for run, f in EARLIER.items()}
assert all(paths.values()), f"attach our dataset with {list(EARLIER.values())}"
a, b = (pd.read_csv(p[0]).set_index("id").label.loc[test.id].to_numpy() for p in paths.values())
# guard: our main-v2 and main-v3 files agree on 98.9% of test rows; far less means a file from another model is attached
assert (a == b).mean() > 0.985, f"{EARLIER} agree on only {(a == b).mean():.1%} of rows: are these our main-v2 and main-v3 files?"
mine = np.array(LABELS)[pred]
vote = np.where(a == b, a, mine)  # when the two earlier runs agree they outvote this one; otherwise this run decides
sub_v = pd.DataFrame({"id": test.id, "label": vote})
assert len(sub_v) == 6000 and sub_v.id.is_unique and sub_v.label.isin(LABELS).all()
sub_v.to_csv("/kaggle/working/submission_vote.csv", index=False)
print(f"vote changed {(vote != mine).sum()} of {len(vote)} rows vs submission.csv | "
      f"test share {(pd.Series(vote).value_counts(normalize=True).reindex(LABELS).round(3).tolist())}")